In [ ]:
#!pip install tqdm
#!pip install statsmodels

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from statsmodels.stats.multitest import multipletests

/home/dcm/env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [3]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 541/541 [00:02<00:00, 250.77it/s]


In [4]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [5]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_Nonprog_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████████| 541/541 [00:00<00:00, 3530.59it/s]


In [6]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [7]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_Prog_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████████| 541/541 [00:00<00:00, 3430.28it/s]


In [8]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [9]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 1741/1741 [00:03<00:00, 574.24it/s]


In [10]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [11]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_Nonprog_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|███████████████| 1741/1741 [00:01<00:00, 1143.57it/s]


In [12]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [13]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_Prog_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|███████████████| 1741/1741 [00:01<00:00, 1168.81it/s]


In [14]:
########
########
########
########
########

In [15]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [16]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████████| 541/541 [00:00<00:00, 2505.96it/s]


In [17]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [18]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_20_year_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████████| 541/541 [00:00<00:00, 3375.19it/s]


In [19]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [20]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_26_year_mgx_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████████| 541/541 [00:00<00:00, 3355.49it/s]


In [21]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [22]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 1741/1741 [00:02<00:00, 756.18it/s]


In [23]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [24]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_20_year_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 1741/1741 [00:01<00:00, 962.74it/s]


In [25]:
otu = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep = '\t')
otu = otu.melt(id_vars = 'file', var_name = 'otu', value_name = 'count')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="otu",
    values="pa",
    fill_value=0
)

In [26]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

results_df.to_csv(f"stats_out/fisher_26_year_mgx_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|███████████████| 1741/1741 [00:01<00:00, 1274.57it/s]
